# DATA INGESTION

In [4]:
from pathlib import Path
from typing import List
import re

import pymupdf
from langchain_core.documents import Document

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from bs4 import BeautifulSoup
from docx import Document as DocxDocument

## File Loaders

In [5]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def load_pdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    normalized_docs = []
    for i, doc in enumerate(docs):
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": doc.metadata.get("page", i),
                    **doc.metadata,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    return normalized_docs


def load_pdf_pymupdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    pdf = pymupdf.open(str(file_path))

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    docs = []
    for page_num, page in enumerate(pdf):
        text = page.get_text("text")
        docs.append(
            Document(
                page_content=clean_text(text),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": page_num,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    pdf.close()
    return docs


def load_txt(file_path: str | Path, encoding: str = "utf-8") -> List[Document]:
    file_path = Path(file_path)
    loader = TextLoader(str(file_path), encoding=encoding)
    docs = loader.load()

    normalized_docs = []
    for doc in docs:
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "txt",
                    **doc.metadata
                }
            )
        )

    return normalized_docs


def load_html(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    # Focus extraction on <main> or <article> if present, else fall back to <body>
    root = soup.find("main") or soup.find("article") or soup.body or soup

    current_heading: str = ""
    lines: list[str] = []

    for tag in root.find_all(["h1", "h2", "h3", "p", "li", "td", "dd"]):
        text = tag.get_text(separator=" ", strip=True)
        if not text:
            continue
        if tag.name in ("h1", "h2", "h3"):
            current_heading = text
        else:
            lines.append(f"[{current_heading}] {text}" if current_heading else text)

    text = "\n".join(lines)

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    return [
        Document(
            page_content=clean_text(text),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "html",
                "title": soup.title.string.strip() if soup.title and soup.title.string else None,
                **({"linked_from": linked_from} if linked_from else {})
            }
        )
    ]

In [6]:
def load_docx(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    doc = DocxDocument(str(file_path))

    # Verify the document has actual content
    if all(not p.text.strip() for p in doc.paragraphs):
        print(f"  Warning: {file_path.name} has no paragraph content, skipping")
        return []

    current_heading: str = ""
    lines: list[str] = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        if para.style.name.startswith("Heading"):
            current_heading = text
        else:
            lines.append(f"[{current_heading}] {text}" if current_heading else text)

    # Extract tables
    for table in doc.tables:
        for row in table.rows:
            cells = " | ".join(cell.text.strip() for cell in row.cells)
            if cells.replace("|", "").strip():
                lines.append(f"[{current_heading}] {cells}" if current_heading else cells)

    linked_from = file_path.parent.name if "linked_docx" in file_path.parts else None

    return [
        Document(
            page_content=clean_text("\n".join(lines)),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "docx",
                **({"linked_from": linked_from} if linked_from else {})
            }
        )
    ]

In [7]:
def load_file(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf_pymupdf(file_path)
    elif suffix in {".txt", ".md"}:
        return load_txt(file_path)
    elif suffix in {".html", ".htm"}:
        return load_html(file_path)
    elif suffix == ".docx":
        return load_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

def load_directory(directory: str | Path) -> List[Document]:
    directory = Path(directory)
    all_docs = []

    for file_path in directory.rglob("*"):
        if file_path.is_file():
            try:
                docs = load_file(file_path)
                all_docs.extend(docs)
                print(f"Loaded: {file_path}")
            except Exception as e:
                print(f"Skipped {file_path}: {e}")

    return all_docs

## Chunking

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


def save_chunks(chunks, path="../data/processed/chunks.json"):
    data = []

    for doc in chunks:
        data.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

## Storing the processed data in json

In [9]:
MIN_CHUNK_SIZE = 100

docs = load_directory("../data/raw")
chunked_docs = text_splitter.split_documents(docs)
chunked_docs = [doc for doc in chunked_docs if len(doc.page_content) >= MIN_CHUNK_SIZE]
for i, doc in enumerate(chunked_docs):
    doc.metadata["chunk_id"] = i

save_chunks(chunked_docs)
print(f"Saved {len(chunked_docs)} chunks")

Loaded: ..\data\raw\inf.elte.hu\en.html
Loaded: ..\data\raw\linked_pdfs\2016_Educational-Plan.pdf
Loaded: ..\data\raw\linked_pdfs\2018_Educational_Plan.pdf
Loaded: ..\data\raw\linked_pdfs\2019_Educational Plan.pdf
Loaded: ..\data\raw\linked_pdfs\2022_Educational-Plan.pdf
Loaded: ..\data\raw\linked_pdfs\2023_Educational-Plan.pdf
Loaded: ..\data\raw\linked_pdfs\2025_Educational_Plan_20250526.pdf
Loaded: ..\data\raw\linked_pdfs\About the BSc in Computer Science programme.pdf
Loaded: ..\data\raw\linked_pdfs\Acceptance of OTDK_TDK work as thesis for MSC in Cartography and MSc in Geoinformatics students.pdf
Loaded: ..\data\raw\linked_pdfs\Acceptance of OTDK_TDK work as thesis for MSC in Computer Sciense Autonomous Systems, Data Science.pdf
Loaded: ..\data\raw\linked_pdfs\Accomplishing the internship with laboratory subjects in the MSc in Computer Science programme.pdf
Loaded: ..\data\raw\linked_pdfs\Adattudomány 2024 ( 2025 október).pdf
Loaded: ..\data\raw\linked_pdfs\Application for the pre